In [ ]:
from openai import OpenAI
import requests
import json

In [ ]:
client = OpenAI()

In [ ]:
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    response.raise_for_status()
    return response.json()

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    response.raise_for_status()
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    response.raise_for_status()
    return response.json()

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록을 가져옵니다. 사용자가 현재 인기 있는 영화나 인기 영화 목록을 물어볼 때 사용합니다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 movie ID에 해당하는 영화의 상세 정보를 가져옵니다. 영화 제목, 줄거리, 평점 등 영화 자체 정보를 물어볼 때 사용합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화 ID"
                    }
                },
                "required": ["id"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 movie ID에 해당하는 영화의 출연진과 제작진 정보를 가져옵니다. 배우, 출연진, 감독, 제작진을 물어볼 때 사용합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화 ID"
                    }
                },
                "required": ["id"]
            },
        },
    },
]

In [ ]:
def check_function_choice(user_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """
너는 Movie Expert Agent다.
사용자의 질문을 읽고 필요한 경우 제공된 함수 중 가장 적절한 함수를 선택한다.

규칙:
- 인기 영화 목록을 물어보면 get_popular_movies를 사용한다.
- 특정 movie ID의 영화 정보/제목/상세내용을 물어보면 get_movie_details를 사용한다.
- 특정 movie ID의 출연진/배우/제작진을 물어보면 get_movie_credits를 사용한다.
"""
            },
            {"role": "user", "content": user_input}
        ],
        tools=tools,
        tool_choice="auto",
    )

    message = response.choices[0].message

    if message.tool_calls:
        for tool_call in message.tool_calls:
            print("선택된 함수명:", tool_call.function.name)
            print("arguments:", tool_call.function.arguments)
    else:
        print("함수 호출 없음")
        print(message.content)

In [ ]:
check_function_choice("지금 인기 있는 영화가 무엇인지 알려줘")

In [ ]:
check_function_choice("movie ID 550에 해당하는 영화가 무엇인지 알려줘")

In [ ]:
check_function_choice("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")